# Module 10 — Scaled Dot-Product Attention

Module 09's neural n-gram model had to pick a fixed context size (3
characters) *before training* — a hard architectural choice, not something
the model could adapt. **Attention** removes that constraint: every
position can look back at *every* earlier position, and the model itself
learns, per query, which earlier positions matter most.

This module builds the mechanism by hand, on tiny toy tensors, with no
training involved yet — just understanding the computation. Module 11
wraps this into multiple parallel heads; Phase 3 assembles it into a full
transformer block.

## 1. Query, Key, Value — the three roles every token plays

For every position in the sequence, we compute three vectors from its
embedding:
- **Query**: "what am I looking for from other positions?"
- **Key**: "what do I contain, for other positions to match against?"
- **Value**: "what do I actually pass along, if someone attends to me?"

A position's query is dotted against *every* position's key to get a
relevance score, and the output is a weighted sum of every position's
*value*, weighted by those scores.

In [ ]:
import math

import torch
import torch.nn.functional as F

torch.manual_seed(42)
seq_len, d_model, d_k = 5, 8, 4

x = torch.randn(seq_len, d_model)  # 5 toy "token embeddings", 8-dim each

# In a real model these would be learned nn.Linear weights; here they're
# fixed random matrices so we can see the mechanism with no training involved.
Wq = torch.randn(d_model, d_k)
Wk = torch.randn(d_model, d_k)
Wv = torch.randn(d_model, d_k)

Q = x @ Wq   # (seq_len, d_k) - one query per position
K = x @ Wk   # (seq_len, d_k) - one key per position
V = x @ Wv   # (seq_len, d_k) - one value per position
print("Q, K, V shapes:", Q.shape, K.shape, V.shape)

## 2. Scores, scaling, and softmax

`Q @ K.T` gives every (query, key) pair's raw relevance score — a
`(seq_len, seq_len)` matrix where entry `[i, j]` is "how much does
position i's query match position j's key." Dividing by `sqrt(d_k)`
prevents these dot products from growing large as `d_k` grows (large
scores would push softmax into a near one-hot, saturated regime with tiny
gradients — this scaling is *why* it's called "scaled" dot-product
attention). Softmax over each row then turns scores into a proper
probability distribution: how much of each value vector to mix in.

In [ ]:
scores = (Q @ K.T) / math.sqrt(d_k)
print("raw scores:\n", scores)

weights_unmasked = F.softmax(scores, dim=-1)
print("\nattention weights (unmasked):\n", weights_unmasked.round(decimals=3))
print("\neach row sums to 1:", torch.allclose(weights_unmasked.sum(dim=-1), torch.ones(seq_len)))

## 3. The causal mask — why language models can't peek at the future

As computed above, position 0 (the first token) is attending to positions
1-4 — tokens that, in an autoregressive language model, **haven't been
generated yet**. Training a next-token predictor this way would let it
"cheat" by looking at the answer. The fix: before softmax, set every
future-position score to `-inf` so softmax assigns it exactly 0
probability. This is a **causal mask** (also called a look-ahead mask).

In [ ]:
# unmasked: position 0 puts real weight on future positions 1-4 - a bug for language modeling
future_weight_unmasked = weights_unmasked[0, 1:].sum().item()
print(f"Unmasked: position 0's attention weight on FUTURE positions: {future_weight_unmasked:.4f} (should be 0 for a causal LM)")

causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
print("\nmask (True = blocked / future position):\n", causal_mask)

scores_masked = scores.masked_fill(causal_mask, float("-inf"))
weights = F.softmax(scores_masked, dim=-1)
print("\nattention weights (causally masked):\n", weights.round(decimals=3))

future_weight_masked = weights[0, 1:].sum().item()
assert future_weight_masked == 0.0
assert torch.allclose(weights.sum(dim=-1), torch.ones(seq_len))
print(f"\nMasked: position 0's attention weight on future positions: {future_weight_masked} - fixed.")

## 4. The output: a weighted mix of Values

In [ ]:
output = weights @ V
print("output shape:", output.shape, "(same shape as V - one context-aware vector per position)")
print(output)

## 5. Wrapping it into one reusable function

This is the exact function Module 11 (multi-head attention) and later
modules will reuse — copied in with this one-line note rather than
re-explained, per this project's convention.

In [ ]:
def scaled_dot_product_attention(Q, K, V, causal=True):
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if causal:
        seq_len_q, seq_len_k = scores.shape[-2], scores.shape[-1]
        mask = torch.triu(torch.ones(seq_len_q, seq_len_k), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights


out2, weights2 = scaled_dot_product_attention(Q, K, V, causal=True)
assert torch.allclose(out2, output)
assert torch.allclose(weights2, weights)
print("Function form matches the step-by-step version above.")

## Recap

- Every position gets a Query, Key, and Value; relevance = scaled dot
  product of Query against every Key; output = softmax-weighted sum of
  Values.
- Scaling by `sqrt(d_k)` keeps the softmax from saturating as dimension
  grows.
- Causal masking (`-inf` on future positions before softmax) is what makes
  this usable for autoregressive language modeling — without it, the model
  would trivially cheat by looking at future tokens.
- This was **one** attention computation over the **whole** embedding.
  Module 11 splits `d_model` into several smaller heads that each run this
  same mechanism in parallel, letting different heads specialize in
  different kinds of relationships.